In [1]:
"""
Summarize per-image features
=============================
Reads per_image_features.csv produced by image_level_analysis.py and generates:

  1. isoform_summary.csv
       Columns = isoforms with >= MIN_IMAGES images
       Rows    = mean_{feat}, median_{feat}, sd_{feat}, mad_{feat}  (interleaved per feature)

  2. protein_difference_summary.csv
       Columns = proteins where BOTH isoforms have >= MIN_IMAGES images
       Rows    = cohens_d_{feat}, p_value_{feat}  (interleaved, per scalar feature)
              + 5 profile-level rows per profile group (granularity, texture_contrast,
                texture_correlation):
                  {profile}_cosine_dist          shape difference (L1-normalised profiles)
                  {profile}_auc_cohens_d         magnitude difference (Cohen's d on AUC)
                  {profile}_auc_pvalue           Mann-Whitney U on per-image AUC
                  {profile}_hotelling_t2         Hotelling T² statistic
                  {profile}_hotelling_pvalue     F-approximation p-value
       Test    = Mann-Whitney U, two-sided (scalar features and AUC)

Usage:
    python summarize_features.py
    python summarize_features.py /path/to/per_image_features.csv
"""

import os, sys
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu, f as f_dist, ks_2samp
import warnings
warnings.filterwarnings('ignore')

In [2]:
# ── USER SETTINGS ──────────────────────────────────────────────────────────────
INPUT_CSV        = './outputs/per_image_features.csv'
OUT_DIR          = os.path.dirname(INPUT_CSV)
ISOFORM_SUFFIXES = ['L', 'S']   # isoform subfolder suffix; names = PROTEIN-L, PROTEIN-S
MIN_IMAGES       = 5           # minimum images per isoform to include

In [6]:
# ── CLI override ───────────────────────────────────────────────────────────────
if len(sys.argv) > 1 and os.path.isfile(sys.argv[1]):
    INPUT_CSV = sys.argv[1]
    OUT_DIR   = os.path.dirname(INPUT_CSV)

# ── Load data ──────────────────────────────────────────────────────────────────
df = pd.read_csv(INPUT_CSV)
print(f"Loaded {INPUT_CSV}: {len(df)} rows, {df['protein'].nunique()} proteins, "
      f"{df['isoform'].nunique()} isoforms")

# Filter out control proteins — controls are handled by control_analysis.py
CONTROL_PROTEINS = ['EGFP-NLS', 'ATXN1', 'G3BP1', 'polyQ']   # edit to match your controls
ctrl_mask = df['protein'].isin(CONTROL_PROTEINS)
if ctrl_mask.sum() > 0:
    print(f"Excluding {ctrl_mask.sum()} control image rows "
          f"({df.loc[ctrl_mask, 'protein'].value_counts().to_dict()})")
    df = df[~ctrl_mask].copy()

META_COLS = ['file', 'protein', 'isoform', 'well_id']
# Also exclude is_control column if present (legacy)
if 'is_control' in df.columns:
    META_COLS.append('is_control')
FEAT_COLS = [c for c in df.columns if c not in META_COLS]
print(f"Features : {len(FEAT_COLS)}")

# ── Profile groups ─────────────────────────────────────────────────────────────
PROFILES = {
    'granularity':        [c for c in FEAT_COLS if c.startswith('granularity_')],
    'texture_contrast':   [c for c in FEAT_COLS if c.startswith('texture_contrast_')],
    'texture_correlation':[c for c in FEAT_COLS if c.startswith('texture_correlation_')],
    'morans_i':           [c for c in FEAT_COLS if c.startswith('morans_i_lag')],
}
# morans_decay is a single derived scalar — kept as its own profile-level row
SCALAR_EXTRAS = ['morans_decay']

# ECDF quantile columns (ecdf_q02 … ecdf_q98, 50 points)
ECDF_COLS = sorted([c for c in FEAT_COLS if c.startswith('ecdf_q')])
HAS_ECDF  = len(ECDF_COLS) > 0
if HAS_ECDF:
    print(f'ECDF columns : {len(ECDF_COLS)} (KS test will be computed)')
else:
    print('ECDF columns : none found — KS test skipped')

# ── Filter: keep only isoforms with >= MIN_IMAGES ──────────────────────────────
iso_counts = df.groupby('isoform').size()
valid_isos = iso_counts[iso_counts >= MIN_IMAGES].index.tolist()
excluded   = iso_counts[iso_counts < MIN_IMAGES]

print(f"\nMin images threshold : {MIN_IMAGES}")
print(f"Isoforms passing     : {len(valid_isos)}")
if len(excluded):
    print(f"Isoforms excluded ({len(excluded)}):")
    for iso, n in excluded.items():
        print(f"  {iso}: {n} images")

df_filt = df[df['isoform'].isin(valid_isos)].copy()

# ── Helper functions ───────────────────────────────────────────────────────────
def mad(x):
    return np.median(np.abs(x - np.median(x)))

def cohens_d(a, b):
    na, nb = len(a), len(b)
    if na < 2 or nb < 2:
        return np.nan
    pooled = np.sqrt(((na-1)*a.std(ddof=1)**2 + (nb-1)*b.std(ddof=1)**2) / (na+nb-2))
    return (b.mean() - a.mean()) / pooled if pooled > 0 else np.nan

def cosine_dist_profiles(mat_a, mat_b):
    """
    Shape difference between two groups of profile vectors.
    Each row is one image's profile. Normalise each row to L1 sum = 1,
    compute per-group mean, return cosine distance between the two means.
    """
    def l1_norm(mat):
        s = mat.sum(axis=1, keepdims=True)
        s[s == 0] = 1          # avoid divide-by-zero for all-zero rows
        return mat / s

    mean_a = l1_norm(mat_a).mean(axis=0)
    mean_b = l1_norm(mat_b).mean(axis=0)
    denom  = np.linalg.norm(mean_a) * np.linalg.norm(mean_b)
    if denom == 0:
        return np.nan
    return float(1.0 - np.dot(mean_a, mean_b) / denom)

def hotelling_t2(mat_a, mat_b):
    """
    Hotelling's T² test for equality of multivariate means.
    Returns (T2_statistic, p_value).
    Uses pseudoinverse if pooled covariance is singular (d >= n).
    Returns (nan, nan) if either group has fewer than d+2 observations.
    """
    n1, d = mat_a.shape
    n2    = mat_b.shape[0]
    if n1 < d + 2 or n2 < d + 2:
        return np.nan, np.nan

    mean_a = mat_a.mean(axis=0)
    mean_b = mat_b.mean(axis=0)
    diff   = mean_a - mean_b

    S_a = np.cov(mat_a, rowvar=False, ddof=1)
    S_b = np.cov(mat_b, rowvar=False, ddof=1)
    S_p = ((n1 - 1) * S_a + (n2 - 1) * S_b) / (n1 + n2 - 2)

    try:
        S_p_inv = np.linalg.inv(S_p)
    except np.linalg.LinAlgError:
        S_p_inv = np.linalg.pinv(S_p)

    T2  = (n1 * n2) / (n1 + n2) * diff @ S_p_inv @ diff
    df1 = d
    df2 = n1 + n2 - d - 1
    if df2 <= 0:
        return float(T2), np.nan
    F   = T2 * df2 / (df1 * (n1 + n2 - 2))
    p   = float(f_dist.sf(F, df1, df2))
    return float(T2), p

def profile_metrics(mat_a, mat_b, profile_name):
    """
    Compute all 5 profile-level metrics for one profile group.
    mat_a, mat_b: (n_images × d_features) float arrays, already dropna'd.
    Returns dict keyed by row name.
    """
    out = {}

    # Shape: cosine distance on L1-normalised mean profiles
    out[f'{profile_name}_cosine_dist'] = cosine_dist_profiles(mat_a, mat_b)

    # Magnitude: AUC = sum across profile dimensions per image
    auc_a = mat_a.sum(axis=1)
    auc_b = mat_b.sum(axis=1)
    out[f'{profile_name}_auc_cohens_d'] = cohens_d(auc_a, auc_b)
    if len(auc_a) >= 2 and len(auc_b) >= 2:
        _, p_auc = mannwhitneyu(auc_a, auc_b, alternative='two-sided')
        out[f'{profile_name}_auc_pvalue'] = float(p_auc)
    else:
        out[f'{profile_name}_auc_pvalue'] = np.nan

    # Omnibus: Hotelling T²
    T2, p_hot = hotelling_t2(mat_a, mat_b)
    out[f'{profile_name}_hotelling_t2']      = T2
    out[f'{profile_name}_hotelling_pvalue']  = p_hot

    return out

def ecdf_ks_metrics(mat_a, mat_b):
    """
    Two ECDF-based metrics comparing intensity distributions between isoforms.

    1. ecdf_quantile_dist: L-infinity distance between the two mean quantile
       functions — max|Q_L(p) - Q_S(p)| over p in [0.02, 0.98].
       Interpretable as the largest intensity difference at any given percentile.
       Sensitive to condensate fraction differences: if L has more condensate
       pixels, its upper quantiles will be higher than S at the same percentile.

    2. ecdf_ks_stat / ecdf_ks_pvalue: two-sample KS test pooling all per-image
       median intensities (one value per image, n = number of replicate images).
       Tests whether the central tendency of the intensity distribution differs.
       Statistically valid unit of observation = image (biological replicate).

    Parameters
    ----------
    mat_a, mat_b : (n_images × 50) float arrays of per-image ECDF quantiles.
    """
    # 1. Quantile L-inf distance between mean quantile functions
    mean_Q_a = mat_a.mean(axis=0)   # shape: (50,) — mean quantile curve for isoform a
    mean_Q_b = mat_b.mean(axis=0)
    quantile_dist = float(np.max(np.abs(mean_Q_a - mean_Q_b)))

    # 2. KS test on per-image median intensity (index 24 ≈ p=0.50 in 0.02..0.98 grid)
    # Using median rather than q95 so the test reflects the full distribution shift,
    # not just the tail. Upper-tail differences are captured by quantile_dist.
    med_idx = mat_a.shape[1] // 2   # index 24 → p ≈ 0.50
    med_a   = mat_a[:, med_idx]
    med_b   = mat_b[:, med_idx]
    if len(med_a) >= 2 and len(med_b) >= 2:
        ks_stat, ks_pval = ks_2samp(med_a, med_b, alternative='two-sided')
    else:
        ks_stat, ks_pval = np.nan, np.nan

    return {
        'ecdf_quantile_dist': quantile_dist,  # max |mean_Q_L(p) - mean_Q_S(p)|
        'ecdf_ks_stat':       ks_stat,         # KS statistic on per-image median intensity
        'ecdf_ks_pvalue':     ks_pval,          # two-sided p-value
    }


# ── 1. Isoform summary ─────────────────────────────────────────────────────────
isoforms  = sorted(valid_isos)
row_index = [f'{stat}_{feat}' for feat in FEAT_COLS
             for stat in ['mean', 'median', 'sd', 'mad']]

iso_data = {}
for iso in isoforms:
    sub = df_filt[df_filt['isoform'] == iso][FEAT_COLS]
    col = {}
    for feat in FEAT_COLS:
        vals = sub[feat].dropna().values
        col[f'mean_{feat}']   = vals.mean()      if len(vals) > 0 else np.nan
        col[f'median_{feat}'] = np.median(vals)  if len(vals) > 0 else np.nan
        col[f'sd_{feat}']     = vals.std(ddof=1) if len(vals) > 1 else np.nan
        col[f'mad_{feat}']    = mad(vals)         if len(vals) > 0 else np.nan
    iso_data[iso] = col

iso_summary = pd.DataFrame(iso_data, index=row_index)
iso_summary.index.name = 'metric_feature'

out_iso = os.path.join(OUT_DIR, 'isoform_summary.csv')
iso_summary.to_csv(out_iso)
print(f"\nSaved {out_iso}  "
      f"({iso_summary.shape[0]} rows × {iso_summary.shape[1]} isoforms)")

# ── 2. Protein difference summary ─────────────────────────────────────────────
protein_names     = sorted(df['protein'].unique())
valid_proteins    = [p for p in protein_names
                     if f'{p}-{ISOFORM_SUFFIXES[0]}' in valid_isos
                     and f'{p}-{ISOFORM_SUFFIXES[1]}' in valid_isos]
excluded_proteins = [p for p in protein_names if p not in valid_proteins]

print(f"\nProteins with both isoforms >= {MIN_IMAGES} images: {len(valid_proteins)}")
if excluded_proteins:
    print(f"Proteins excluded: {excluded_proteins}")

# Row index: profile-level metrics only (no per-feature scalar rows)
profile_row_names = [f'{pname}_{metric}'
                     for pname in PROFILES
                     for metric in ['cosine_dist', 'auc_cohens_d', 'auc_pvalue',
                                    'hotelling_t2', 'hotelling_pvalue']]
# morans_decay: single scalar — stored as cohens_d / p_value pair
scalar_extra_names = [f'{stat}_{feat}'
                      for feat in SCALAR_EXTRAS
                      for stat in ['cohens_d', 'p_value']]
ks_row_names   = ['ecdf_quantile_dist', 'ecdf_ks_stat', 'ecdf_ks_pvalue'] if HAS_ECDF else []
row_index_diff = profile_row_names + scalar_extra_names + ks_row_names

diff_data = {}
for protein in valid_proteins:
    iso1 = f'{protein}-{ISOFORM_SUFFIXES[0]}'
    iso2 = f'{protein}-{ISOFORM_SUFFIXES[1]}'
    sub1 = df_filt[df_filt['isoform'] == iso1][FEAT_COLS]
    sub2 = df_filt[df_filt['isoform'] == iso2][FEAT_COLS]

    col = {}

    # Profile-level stats
    for pname, pcols in PROFILES.items():
        # Drop rows where any profile feature is NaN
        mat1 = sub1[pcols].dropna().values.astype(float)
        mat2 = sub2[pcols].dropna().values.astype(float)
        if len(mat1) >= 2 and len(mat2) >= 2:
            col.update(profile_metrics(mat1, mat2, pname))
        else:
            for metric in ['cosine_dist', 'auc_cohens_d', 'auc_pvalue',
                           'hotelling_t2', 'hotelling_pvalue']:
                col[f'{pname}_{metric}'] = np.nan

    # Scalar extras (morans_decay)
    for feat in SCALAR_EXTRAS:
        a = sub1[feat].dropna().values if feat in sub1.columns else np.array([])
        b = sub2[feat].dropna().values if feat in sub2.columns else np.array([])
        if len(a) >= 2 and len(b) >= 2:
            _, p = mannwhitneyu(a, b, alternative='two-sided')
            d    = cohens_d(a, b)
        else:
            d, p = np.nan, np.nan
        col[f'cohens_d_{feat}'] = d
        col[f'p_value_{feat}']  = p

    # ECDF KS test
    if HAS_ECDF:
        ecdf1 = sub1[ECDF_COLS].dropna().values.astype(float)
        ecdf2 = sub2[ECDF_COLS].dropna().values.astype(float)
        if len(ecdf1) >= 2 and len(ecdf2) >= 2:
            col.update(ecdf_ks_metrics(ecdf1, ecdf2))
        else:
            col.update({'ecdf_quantile_dist': np.nan, 'ecdf_ks_stat': np.nan, 'ecdf_ks_pvalue': np.nan})

    diff_data[protein] = col

# ── EGFP-NLS vs ATXN1 control comparison (appended as pseudo-protein) ────────
# Controls are stored as protein='control', isoform=<name>.
# Pull directly from df (not df_filt, which only contains experimental isoforms).
EGFP_ISO  = 'EGFP-NLS'
ATXN1_ISO = 'ATXN1'
df_egfp  = df[df['isoform'] == EGFP_ISO][FEAT_COLS]
df_atxn1 = df[df['isoform'] == ATXN1_ISO][FEAT_COLS]

if len(df_egfp) >= 2 and len(df_atxn1) >= 2:
    ctrl_col = {}
    for pname, pcols in PROFILES.items():
        mat_e = df_egfp[pcols].dropna().values.astype(float)
        mat_a = df_atxn1[pcols].dropna().values.astype(float)
        if len(mat_e) >= 2 and len(mat_a) >= 2:
            ctrl_col.update(profile_metrics(mat_e, mat_a, pname))
        else:
            for metric in ['cosine_dist', 'auc_cohens_d', 'auc_pvalue',
                           'hotelling_t2', 'hotelling_pvalue']:
                ctrl_col[f'{pname}_{metric}'] = np.nan
    for feat in SCALAR_EXTRAS:
        a = df_egfp[feat].dropna().values if feat in df_egfp.columns else np.array([])
        b = df_atxn1[feat].dropna().values if feat in df_atxn1.columns else np.array([])
        if len(a) >= 2 and len(b) >= 2:
            _, p = mannwhitneyu(a, b, alternative='two-sided')
            d    = cohens_d(a, b)
        else:
            d, p = np.nan, np.nan
        ctrl_col[f'cohens_d_{feat}'] = d
        ctrl_col[f'p_value_{feat}']  = p
    if HAS_ECDF:
        ecdf_e = df_egfp[ECDF_COLS].dropna().values.astype(float)
        ecdf_a = df_atxn1[ECDF_COLS].dropna().values.astype(float)
        if len(ecdf_e) >= 2 and len(ecdf_a) >= 2:
            ctrl_col.update(ecdf_ks_metrics(ecdf_e, ecdf_a))
        else:
            ctrl_col.update({'ecdf_quantile_dist': np.nan,
                             'ecdf_ks_stat': np.nan, 'ecdf_ks_pvalue': np.nan})
    diff_data['EGFP-NLS_vs_ATXN1'] = ctrl_col
    print(f'Added EGFP-NLS vs ATXN1 control comparison '
          f'(n_EGFP={len(df_egfp)}, n_ATXN1={len(df_atxn1)})')
else:
    print('WARNING: EGFP-NLS or ATXN1 not found in CSV — control comparison skipped.')

diff_summary = pd.DataFrame(diff_data, index=row_index_diff)
diff_summary.index.name = 'metric_feature'

out_diff = os.path.join(OUT_DIR, 'protein_difference_summary.csv')
diff_summary.T.to_csv(out_diff)
print(f"Saved {out_diff}  "
      f"({diff_summary.shape[0]} rows × {diff_summary.shape[1]} proteins)")
print(f"  — {len(profile_row_names)} profile-level rows "
      f"({len(PROFILES)} profiles × 5 metrics each)")
print(f"  — {len(scalar_extra_names)} scalar-extra rows "
      f"({SCALAR_EXTRAS}  ×  cohens_d / p_value)")
if HAS_ECDF:
    print(f"  — {len(ks_row_names)} ECDF rows (quantile_dist, ks_stat, ks_pvalue)")

# ── Console preview ────────────────────────────────────────────────────────────
all_rows = profile_row_names + scalar_extra_names + ks_row_names
print("\nProfile-level metrics (all proteins, incl. EGFP-NLS_vs_ATXN1):")
print(f"  {'Metric':<45} {'min':>8} {'median':>8} {'max':>8}")
print("  " + "─" * 73)
for row in all_rows:
    if row in diff_summary.index:
        vals = diff_summary.loc[row].dropna().values
        if len(vals):
            print(f"  {row:<45} {vals.min():>8.4f} {np.median(vals):>8.4f} {vals.max():>8.4f}")

print("\nDone.")

Loaded ./outputs/per_image_features.csv: 26694 rows, 101 proteins, 213 isoforms
Features : 81
ECDF columns : 50 (KS test will be computed)

Min images threshold : 5
Isoforms passing     : 213

Saved ./outputs/isoform_summary.csv  (324 rows × 213 isoforms)

Proteins with both isoforms >= 5 images: 99
Proteins excluded: ['CACNA2D1', 'control']
Added EGFP-NLS vs ATXN1 control comparison (n_EGFP=92, n_ATXN1=286)
Saved ./outputs/protein_difference_summary.csv  (25 rows × 100 proteins)
  — 20 profile-level rows (4 profiles × 5 metrics each)
  — 2 scalar-extra rows (['morans_decay']  ×  cohens_d / p_value)
  — 3 ECDF rows (quantile_dist, ks_stat, ks_pvalue)

Profile-level metrics (all proteins, incl. EGFP-NLS_vs_ATXN1):
  Metric                                             min   median      max
  ─────────────────────────────────────────────────────────────────────────
  granularity_cosine_dist                         0.0000   0.0014   0.0862
  granularity_auc_cohens_d                       -3